---
title: Plotting xarray data from the Open Data Cube
short_title: Plotting
subject: Beginner Guide
subtitle: Visualising xarray Datasets and DataArrays returned by dc.load.
description: Visualising xarray Datasets and DataArrays returned by dc.load.
authors:
  - name: Muhammad Taufik
    github: taufik-shf
  - name: Alex G Leith
    github: alexgleith
keywords:
  - open-data-cube
  - odc
  - xarray
  - plotting
  - beginner-guide
---

This notebook covers how to plot xarray data returned by `dc.load`, working from single-band images up to multi-band composites and facet grids for comparing multiple panels at once.[^edits]

[^edits]: Tutorial notebooks update automatically; edits to a tutorial notebook may be overwritten on the next update. Keep a working copy in a separate file to preserve changes.

## A. Objectives

- Plot a single band as a colour-mapped image
- Stack red, green, and blue into a true-colour composite
- Split a band across time into a row of panels

## B. Loading a sample dataset

Start loading the data to plot.

In [ ]:
from datacube import Datacube

dc = Datacube(app="plotting")

query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": ("2024", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

ds = dc.load(**query)

## C. Plotting

Plotting is one way to visualise the data at hand, and often the fastest way to make sense of what the pixel values contain. Earth observation benefits especially from the Jupyter notebook environment, where a plot of the data or of an analysis result appears right beneath the code that produced it.

Since `dc.load` returns an `xarray.Dataset` and plotting fundamentally works with 2D data, this notebook focuses on how to handle the data before plotting it under several scenarios.

## D. Single-band image

The first scenario is plotting a single band. The loaded data contains multiple bands and time slices, so a single band and time slice need to be selected first, as shown in the example below.

In [ ]:
ds.red.isel(time=0).plot()

The code chains three operations. First `ds.red` selects the red band, then `.isel(time=0)` narrows it down to the first of the two loaded time slices (2024), and finally `.plot()` renders the result as the image above. The colourbar comes with every plot by default.

The `cmap` argument accepts any of matplotlib's built-in colourmaps, listed in the [matplotlib colormap reference](https://matplotlib.org/stable/gallery/color/colormap_reference.html). For example, `"Reds"` produces a sequential red palette:

In [ ]:
ds.red.isel(time=0).plot(cmap="Reds")

## E. Multi-band composite

Visualising the true colour as observed by the satellite requires combining the red, green, and blue bands into a single image before plotting, a process known as compositing. The code below shows how.

In [ ]:
rgb = ds[["red", "green", "blue"]].to_array(dim="band").isel(time=0)
rgb

The RGB composite is built by first selecting `ds[["red", "green", "blue"]]` from the Dataset, returning a smaller Dataset with just those three bands. The `.to_array` method then stacks the three variables into a single DataArray along a new dimension, and the `dim="band"` argument specifies the name of that new dimension. The `dim` argument is optional. Without it, the new dimension takes the default name `"variable"`, and any other name works equally well. With the bands stacked, `.isel(time=0)` picks the first time slice (2024) by index, matching the pattern used earlier for the single-band plot. The result is stored in `rgb`, which is a 3D array with dimensions `(band, y, x)`.

With `rgb` in hand, calling `.plot()` directly seems like the obvious next step:

In [ ]:
rgb.plot()

However, it returns a histogram and not the image intended. In the single-band example, `.plot()` worked because the DataArray had exactly two dimensions (`y` and `x`), and xarray knew to draw that as a colour-mapped image. Here `rgb` has three dimensions (`band`, `y`, and `x`), and xarray cannot infer which pair forms the image plane and which holds the colour channels. Without that information, `.plot()` falls back to a histogram of all values.

In this kind of scenario, `.plot.imshow()` is used instead:

In [ ]:
rgb.plot.imshow(vmin=0, vmax=3000)

`.plot.imshow()` treats the length-3 `band` dimension as the colour channels and draws the composite as an image.

The `vmin` and `vmax` arguments set the display stretch. Values at or below `vmin` render as pure black in each channel, and values at or above `vmax` render as fully saturated.

There are public resources documenting common stretches to use for each product. The `s2_geomad_annual` product used here is derived from Sentinel-2 surface reflectance, where 0 to 3000 is a commonly used stretch. Meanwhile, Landsat surface reflectance commonly uses 0 to 7500. Try running the plot without `vmin` and `vmax` to see how the stretch affects the visualisation.

## F. Facet grid

Comparing observations across different points in time is another common visualisation need. When several images sit side by side within a grid of panels, differences between them become easy to spot without switching between separate plots. xarray supports this pattern through faceting, achieved by passing the `col=` parameter to the `.plot()` method:

In [ ]:
ds.red.plot(col="time", cmap="Reds", vmin=0, vmax=3000)

Adding `col="time"` tells `.plot()` to facet along the `time` dimension, with each panel holding a 2D `(y, x)` slice. The `cmap`, `vmin`, and `vmax` arguments apply consistently to every panel, which is what makes them directly comparable: the same pixel value renders the same colour in every panel.

The same faceting pattern works on any coordinate dimension. Faceting on `band` produces one panel per band:

In [ ]:
rgb.plot(col="band", vmin=0, vmax=3000)

## G. Next steps

Continue to [`06_basic_analysis.ipynb`](./06_basic_analysis.ipynb).